In [ ]:
import random, math

def direct_disks_box(N, sigma):
    '''
    Esta función me genera un microestado de un macroestado definido por N discos de radio sigma
    en un recuadro de [0,1]x[0,1].

    La idea consiste en tener inicialmente el espacio vacío. Luego, voy poniendo aleatoriamente
    los diferentes discos uno por uno (run) y los voy guardando en mi espacio. Si alguno de estos
    discos se sobrelapa con otro, elimino mi espacio y vuelvo a empezar desde cero. Sigo de esta
    forma hasta que logro hacer una run completa sin ningún error. Este espacio corresponde a un
    microestado de mi macroestado.
    '''
    condition = False  # Defino una variable booleana que mantendrá mi run hasta que se logre generar una configuración aceptable

    while condition == False:
        L = [[random.uniform(sigma, 1.0 - sigma),
              random.uniform(sigma, 1.0 - sigma)]]  # Represento mi espacio como L, que es un vector con las coordenadas de mis discos, e inicio poniendo un disco en el espacio de forma aleatoria

        for k in range(1, N):  # Inicio mi run poniendo disco por disco
            a = (random.uniform(sigma, 1.0 - sigma),
                 random.uniform(sigma, 1.0 - sigma))  # Genero una posición aleatoria

            min_dist = min(
                math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)
                for b in L
            )  # Calculo las distancias de mi nuevo disco respecto a todos los demás discos en el espacio y me quedo con la distancia mínima

            if min_dist < 2.0 * sigma:  # Reviso la condición: si la distancia a alguno de los centros de los discos es menor que un diámetro, entonces tengo sobrelapamiento, elimino todo mi progreso y vuelvo a empezar
                condition = False
                break
            else:  # Si la distancia mínima es mayor o igual a un diámetro, pongo el nuevo disco en el espacio
                L.append(a)
                condition = True

    return L


# Voy a demostrar que este algoritmo mantiene la equiprobabilidad.
# La idea básicamente es tomar 3 configuraciones y luego generar muchos microestados con la función direct_disks_box.
# Después, observo cuántos de los microestados generados se aproximan a mis tres microestados de referencia
# y verifico que las cantidades de aproximaciones sean muy parecidas.


# Defino mis valores iniciales
sigma = 0.15

del_xy = 0.05  # Esta será mi tolerancia para definir si un microestado generado se aproxima o no a los examinados

n_runs = 10000  # Número de microestados que genero


# Microestados de prueba, que son los que voy a analizar
conf_a = ((0.30, 0.30), (0.30, 0.70), (0.70, 0.30), (0.70, 0.70))
conf_b = ((0.20, 0.20), (0.20, 0.80), (0.75, 0.25), (0.75, 0.75))
conf_c = ((0.30, 0.20), (0.30, 0.80), (0.70, 0.20), (0.70, 0.70))


configurations = [conf_a, conf_b, conf_c]  # Vector con mis configuraciones

hits = {conf_a: 0, conf_b: 0, conf_c: 0}  # Cuenta el número de microestados generados que se aproximan a los que estoy examinando


for run in range(n_runs):
    x_vec = direct_disks_box(4, sigma)  # Microestado generado de 4 discos

    for conf in configurations:  # Comparo mi microestado generado con los que voy a examinar
        condition_hit = True  # Esta será mi variable que me dirá si el microestado generado se aproxima o no al estado examinado

        '''
        Mido las distancias componente a componente de todos los discos del microestado examinado
        respecto a los discos del microestado generado. Para cada disco de referencia me quedo con
        la distancia mínima y reviso si cumple con mi tolerancia.

        Si existe al menos un disco generado que está lo suficientemente cerca de cada uno de los
        discos del microestado examinado, digo que cumple la condición y, por tanto, considero que
        este microestado generado es aproximadamente igual al de referencia.
        '''

        for b in conf:
            condition_b = min(
                max(abs(a[0] - b[0]), abs(a[1] - b[1]))
                for a in x_vec
            ) < del_xy

            condition_hit *= condition_b  # Existe al menos un disco en x_vec que está lo suficientemente cerca de b

        if condition_hit:
            hits[conf] += 1  # Tengo una aproximación


for conf in configurations:
    print(conf, hits[conf])

((0.3, 0.3), (0.3, 0.7), (0.7, 0.3), (0.7, 0.7)) 0
((0.2, 0.2), (0.2, 0.8), (0.75, 0.25), (0.75, 0.75)) 3
((0.3, 0.2), (0.3, 0.8), (0.7, 0.2), (0.7, 0.7)) 1


In [ ]:
import random

L = [[0.25, 0.25], [0.75, 0.25], [0.25, 0.75], [0.75, 0.75]]

'''
La idea de este algoritmo es que, dada una configuración inicial correcta, muevo aleatoriamente sus discos.
Cada movimiento en el que el disco no se sobrelape con otro ni se salga del espacio representa una nueva
configuración.

Por lo tanto, si sigo moviendo los discos durante muchos pasos, puedo llegar en n_steps a una configuración
que esté aproximadamente descorrelacionada de la configuración inicial. De esta forma, la cadena puede
recorrer diferentes microestados permitidos del sistema y mantener la equiprobabilidad de las configuraciones.
'''

sigma = 0.15  # Radio
sigma_sq = sigma ** 2

delta = 0.1  # Tamaño máximo de cada paso que pueden dar los discos

n_steps = 1000  # Número de pasos


for steps in range(n_steps):

    a = random.choice(L)  # Escojo aleatoriamente uno de los discos de la configuración actual y lo intento mover

    b = [
        a[0] + random.uniform(-delta, delta),
        a[1] + random.uniform(-delta, delta)
    ]  # b representa la nueva posición propuesta para el disco seleccionado

    min_dist = min(
        (b[0] - c[0]) ** 2 + (b[1] - c[1]) ** 2
        for c in L if c != a
    )  # Mido la mínima distancia al cuadrado entre el disco movido y el resto de los discos

    box_cond = min(b[0], b[1]) < sigma or max(b[0], b[1]) > 1.0 - sigma  # Mido si el movimiento provocó que el disco se saliera del espacio respecto a x o y

    if not (box_cond or min_dist < 4.0 * sigma ** 2):  # Como estoy usando la distancia al cuadrado, comparo con (2*sigma)^2 = 4*sigma^2
        a[:] = b  # Acepto el movimiento y actualizo la configuración

print(L)  # Devuelvo la configuración obtenida después de n_steps, que debería ser aproximadamente independiente de la inicial

[[0.3544510997925344, 0.34142312469356495], [0.7590384674240165, 0.17072290106660565], [0.4677581365469628, 0.8368434119672423], [0.8101820636464238, 0.7915950273938817]]


## Tiempo de colisión

La posición de cada disco evoluciona mediante movimiento rectilíneo uniforme:

$$
\mathbf{x}_k(t)
=
\mathbf{x}_k(t_0)
+
\mathbf{v}_k(t-t_0),
$$

$$
\mathbf{x}_l(t)
=
\mathbf{x}_l(t_0)
+
\mathbf{v}_l(t-t_0).
$$

Definimos la posición relativa y la velocidad relativa como

$$
\Delta \mathbf{x}
=
\mathbf{x}_k(t_0)-\mathbf{x}_l(t_0),
$$

$$
\Delta \mathbf{v}
=
\mathbf{v}_k-\mathbf{v}_l.
$$

Entonces, la posición relativa en el tiempo es

$$
\Delta\mathbf{x}(t)
=
\Delta\mathbf{x}
+
\Delta\mathbf{v}(t-t_0).
$$

Dos discos de radio $\sigma$ chocan cuando la distancia entre sus centros es igual a $2\sigma$:

$$
|\Delta\mathbf{x}(t)|=2\sigma.
$$

Elevando al cuadrado,

$$
\left|
\Delta\mathbf{x}
+
\Delta\mathbf{v}(t-t_0)
\right|^2
=
4\sigma^2.
$$

Definiendo

$$
\tau=t-t_0,
$$

obtenemos

$$
(\Delta\mathbf{v}\cdot\Delta\mathbf{v})\tau^2
+
2(\Delta\mathbf{x}\cdot\Delta\mathbf{v})\tau
+
(\Delta\mathbf{x}\cdot\Delta\mathbf{x}-4\sigma^2)
=
0.
$$

Para simplificar, definimos

$$
a=\Delta\mathbf{v}\cdot\Delta\mathbf{v},
$$

$$
b=\Delta\mathbf{x}\cdot\Delta\mathbf{v},
$$

$$
c=\Delta\mathbf{x}\cdot\Delta\mathbf{x}-4\sigma^2.
$$

La ecuación queda

$$
a\tau^2+2b\tau+c=0.
$$

### Condición para que los discos se estén acercando

La distancia relativa al cuadrado entre los discos está dada por

$$
d^2(\tau)
=
\left|
\Delta\mathbf{x}
+
\Delta\mathbf{v}\tau
\right|^2.
$$

Derivando respecto a $\tau$,

$$
\frac{d}{d\tau}d^2(\tau)
=
2\left(
\Delta\mathbf{x}
+
\Delta\mathbf{v}\tau
\right)\cdot\Delta\mathbf{v}.
$$

En el instante inicial, $\tau=0$, tenemos

$$
\left.
\frac{d}{d\tau}d^2(\tau)
\right|_{\tau=0}
=
2\Delta\mathbf{x}\cdot\Delta\mathbf{v}.
$$

Como

$$
b=\Delta\mathbf{x}\cdot\Delta\mathbf{v},
$$

entonces

$$
\left.
\frac{d}{d\tau}d^2(\tau)
\right|_{\tau=0}
=
2b.
$$

Por lo tanto, si

$$
b<0,
$$

entonces

$$
\frac{d}{d\tau}d^2<0,
$$

lo que significa que la distancia entre los discos está disminuyendo y, por tanto, los discos se están acercando.

Por el contrario, si

$$
b>0,
$$

la distancia está aumentando y los discos se están alejando.

### Tiempo de colisión entre dos discos

Resolviendo la ecuación cuadrática,

$$
\tau_{1,2}
=
\frac{-b\pm\sqrt{b^2-ac}}{a}.
$$

El tiempo hasta la primera colisión futura entre los discos es

$$
\boxed{
\tau_{\mathrm{col}}
=
\frac{-b-\sqrt{b^2-ac}}{a}
}
$$

y, por tanto, el tiempo absoluto de colisión es

$$
\boxed{
t_{\mathrm{col}}
=
t_0+
\frac{-b-\sqrt{b^2-ac}}{a}
}.
$$

Esta colisión es posible siempre que

$$
b<0,
\qquad
b^2-ac\geq0,
\qquad
\tau_{\mathrm{col}}>0.
$$

Estas condiciones significan:

$$
b<0
\qquad
\Longrightarrow
\qquad
\text{los discos se están acercando},
$$

$$
b^2-ac\geq0
\qquad
\Longrightarrow
\qquad
\text{existe una solución real para la colisión},
$$

y

$$
\tau_{\mathrm{col}}>0
\qquad
\Longrightarrow
\qquad
\text{la colisión ocurre en el futuro}.
$$


## Tiempo de colisión con una pared

El tiempo que tarda un disco en llegar a una pared está dado por

$$
\Delta t_{\mathrm{pared}}
=
\begin{cases}
\dfrac{1-\sigma-x_a}{v_a}, & v_a>0,\\[8pt]
\dfrac{x_a-\sigma}{|v_a|}, & v_a<0,\\[8pt]
\infty, & v_a=0.
\end{cases}
$$

donde $x_a$ representa la posición del centro del disco en una de las
coordenadas y $v_a$ representa la componente correspondiente de su velocidad.

Si $v_a>0$, el disco se mueve hacia la pared ubicada en

$$
x=1-\sigma.
$$

Si $v_a<0$, el disco se mueve hacia la pared ubicada en

$$
x=\sigma.
$$

Finalmente, si $v_a=0$, el disco no se mueve en esa coordenada y, por tanto,

$$
\Delta t_{\mathrm{pared}}=\infty.
$$

In [8]:
'''
Estas dos funciones calculan los tiempos necesarios para que ocurra un evento, como el choque
de un disco con una pared o la colisión entre dos discos. En el caso del muestreo basado en eventos,
la idea es generar microestados aproximadamente independientes haciendo evolucionar el sistema
inicial en el tiempo.
'''

def wall_time(pos_a, vel_a, sigma):  # Esta función calcula el tiempo que le toma al disco llegar a chocar con una pared

    if vel_a > 0.0:  # Miro la dirección de la velocidad, suponiendo positiva la dirección creciente del eje

        del_t = (1.0 - sigma - pos_a) / vel_a  # Usando movimiento rectilíneo uniforme despejo el tiempo. En este caso, calculo qué tan lejos está el disco de la pared ubicada en 1 - sigma

    elif vel_a < 0.0:  # Miro la dirección de la velocidad

        del_t = (pos_a - sigma) / abs(vel_a)  # Si el disco se mueve en el sentido contrario, calculo qué tan lejos está de la pared ubicada en sigma

    else:

        del_t = float('inf')  # Si no hay velocidad en esta dirección, el disco nunca llegará a esa pared

    return del_t


def pair_time(pos_a, vel_a, pos_b, vel_b, sigma):  # Esta función calcula el tiempo necesario para que dos discos choquen entre sí

    del_x = [pos_b[0] - pos_a[0], pos_b[1] - pos_a[1]]  # Calculo la posición relativa entre los dos discos

    del_x_sq = del_x[0] ** 2 + del_x[1] ** 2  # Calculo la norma al cuadrado de la posición relativa


    del_v = [vel_b[0] - vel_a[0], vel_b[1] - vel_a[1]]  # Calculo la velocidad relativa entre los dos discos

    del_v_sq = del_v[0] ** 2 + del_v[1] ** 2  # Calculo la norma al cuadrado de la velocidad relativa


    scal = del_v[0] * del_x[0] + del_v[1] * del_x[1]  # Calculo el producto punto entre la velocidad relativa y la posición relativa


    Upsilon = scal ** 2 - del_v_sq * (del_x_sq - 4.0 * sigma ** 2)  # Calculo el discriminante de la ecuación cuadrática para el tiempo de colisión


    if Upsilon > 0.0 and scal < 0.0:  # Reviso las condiciones necesarias para que tenga sentido físico una futura colisión: debe existir una solución real y los discos deben estar acercándose

        del_t = -(scal + math.sqrt(Upsilon)) / del_v_sq  # La ecuación produce dos tiempos posibles. Escojo la raíz menor positiva, que corresponde al primer instante en el que los discos entran en contacto

    else:

        del_t = float('inf')  # Si no cumplen las condiciones de colisión, considero que el tiempo de colisión es infinito

    return del_t

En el modelo de **discos duros lisos y sin fricción**, la interacción entre dos discos depende únicamente de la distancia entre sus centros.

Si las posiciones son

$$
\mathbf r_a,\qquad \mathbf r_b,
$$

la separación relativa es

$$
\mathbf r=\mathbf r_b-\mathbf r_a,
$$

y la distancia entre centros es

$$
r=|\mathbf r|.
$$

Si la interacción se describe mediante un potencial central

$$
U=U(r),
$$

la fuerza está dada por

$$
\mathbf F=-\nabla U.
$$

Como $U$ depende únicamente de $r$, entonces

$$
\boxed{
\mathbf F
=
-\frac{dU}{dr}\,\hat{\mathbf r}
}
$$

donde

$$
\hat{\mathbf r}
=
\frac{\mathbf r}{|\mathbf r|}.
$$

Por tanto, la fuerza solo tiene componente normal:

$$
\boxed{
\mathbf F=F_n\hat{\mathbf n}
}
$$

y no existe componente tangencial:

$$
\boxed{
F_t=0
}.
$$
Como la fuerza durante la colisión solo tiene componente normal,

$$
\mathbf F = F_n \hat{\mathbf n},
$$

la componente tangencial de la velocidad no cambia.

Podemos descomponer la velocidad como

$$
\mathbf v
=
\mathbf v_n
+
\mathbf v_t,
$$

donde

$$
\mathbf v_n
=
(\mathbf v\cdot\hat{\mathbf n})\hat{\mathbf n},
$$

y

$$
\mathbf v_t
=
\mathbf v-\mathbf v_n.
$$

Después de la colisión,

$$
\boxed{
\mathbf v_t'=\mathbf v_t
}
$$

mientras que la componente normal sí cambia:

$$
\boxed{
\mathbf v_n'\neq\mathbf v_n
}.
$$

Por tanto,

$$
\boxed{
\mathbf v'
=
\mathbf v_n'
+
\mathbf v_t
}
$$

es decir, la colisión modifica únicamente la componente normal de la velocidad.
Definimos el vector que une los centros de los discos como

$$
\Delta \mathbf r
=
\mathbf r_b-\mathbf r_a.
$$

La dirección normal de la colisión está dada por

$$
\hat{\mathbf n}
=
\frac{\Delta\mathbf r}{|\Delta\mathbf r|}.
$$

Para discos de masas iguales, las velocidades después de la colisión elástica son

$$
\boxed{
\mathbf v_a'
=
\mathbf v_a
+
\left[
(\mathbf v_b-\mathbf v_a)\cdot\hat{\mathbf n}
\right]
\hat{\mathbf n}
}
$$

y

$$
\boxed{
\mathbf v_b'
=
\mathbf v_b
-
\left[
(\mathbf v_b-\mathbf v_a)\cdot\hat{\mathbf n}
\right]
\hat{\mathbf n}
}.
$$

In [2]:
"""
En el muestreo basado en eventos, el sistema es completamente determinista a nivel teórico.
Esto significa que, si conozco exactamente las posiciones y las velocidades de todas las
partículas, sin ninguna incertidumbre, podría invertir la evolución temporal y regresar
a la condición inicial.

Sin embargo, en la práctica las posiciones y las velocidades siempre tienen una precisión
finita. Debido al carácter caótico de la dinámica, pequeñas diferencias o errores en estas
cantidades pueden crecer rápidamente a medida que aumenta el número de eventos.

Por esta razón, después de un tiempo suficientemente largo, el sistema pierde de manera
estadística la memoria de la condición inicial. Esto no significa que la información inicial
desaparezca de las ecuaciones, sino que las configuraciones obtenidas dejan de estar
fuertemente correlacionadas con ella.

Por lo tanto, después de un número suficientemente grande de eventos, puedo considerar
que el microestado generado está aproximadamente descorrelacionado de la condición inicial
y puede utilizarse como una muestra representativa del sistema en equilibrio.
"""


# Defino manualmente tres configuraciones espaciales de referencia
conf_a = ((0.30, 0.30), (0.30, 0.70), (0.70, 0.30), (0.70, 0.70))
conf_b = ((0.20, 0.20), (0.20, 0.80), (0.75, 0.25), (0.75, 0.75))
conf_c = ((0.30, 0.20), (0.30, 0.80), (0.70, 0.20), (0.70, 0.70))

configurations = [conf_a, conf_b, conf_c]  # Agrupo las tres configuraciones de referencia que quiero buscar

hits = {conf_a: 0, conf_b: 0, conf_c: 0}  # Cuento cuántas configuraciones simuladas se aproximan a cada configuración de referencia

del_xy = 0.10  # Tolerancia en la norma infinito para comparar dos posiciones


# Construyo las condiciones iniciales para mi algoritmo
pos = [[0.25, 0.25], [0.75, 0.25], [0.25, 0.75], [0.75, 0.75]]  # Defino las posiciones iniciales de los centros de los cuatro discos

vel = [[0.21, 0.12], [0.71, 0.18], [-0.23, -0.79], [0.78, 0.1177]]  # Defino las velocidades iniciales de los cuatro discos

singles = [(0, 0), (0, 1), (1, 0), (1, 1), (2, 0), (2, 1), (3, 0), (3, 1)]  # Pares (disco, coordenada): 0 representa x y 1 representa y

pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]  # Todas las parejas distintas de discos, sin repeticiones ni permutaciones

sigma = 0.10  # Defino el radio de los discos

t = 0.0  # Defino el tiempo inicial

n_events = 5000000  # Número de eventos de colisión que se simularán


for event in range(n_events):

    '''
    Dejo evolucionar libremente el sistema hasta la siguiente colisión.
    En el instante del evento actualizo las velocidades y continúo
    la evolución hasta el siguiente evento.
    '''


    # Calculo el tiempo hasta cada posible colisión
    wall_times = [wall_time(pos[k][l], vel[k][l], sigma) for k, l in singles]  # Calculo los tiempos necesarios para que cada disco choque con una pared

    pair_times = [pair_time(pos[k], vel[k], pos[l], vel[l], sigma) for k, l in pairs]  # Calculo los tiempos necesarios para las posibles colisiones entre pares de discos

    # Las listas conservan el mismo orden que singles y pairs, por lo que puedo recuperar posteriormente qué disco o qué pareja produjo el siguiente evento

    next_event = min(wall_times + pair_times)  # Encuentro el menor tiempo entre todas las posibles colisiones; este será el tiempo hasta el siguiente evento

    t_previous = t  # Guardo el tiempo actual para poder avanzar posteriormente las posiciones por intervalos


    for inter_times in range(int(t + 1), int(t + next_event + 1)):  # Recorro los tiempos enteros comprendidos entre el evento actual y la siguiente colisión

        del_t = inter_times - t_previous  # Calculo el intervalo de tiempo necesario para llegar desde el último tiempo procesado hasta el tiempo entero actual

        for k, l in singles:  # Actualizo las componentes x e y de todos los discos mediante movimiento rectilíneo uniforme

            pos[k][l] += vel[k][l] * del_t  # Entre colisiones la velocidad permanece constante

        t_previous = inter_times  # Guardo el último tiempo entero hasta el cual he evolucionado el sistema


    """
    Como se explicó anteriormente, el sistema necesita un tiempo suficientemente largo
    para perder estadísticamente la memoria de la condición inicial.

    En este algoritmo no se introduce explícitamente un tiempo de termalización antes
    de comenzar las mediciones. En su lugar, se utiliza un número muy grande de eventos,
    de manera que se espera que la contribución del transitorio inicial sea pequeña
    frente al número total de medidas.

    Además, si queremos obtener microestados aproximadamente independientes entre sí,
    también deberíamos dejar transcurrir un cierto tiempo de descorrelación entre
    mediciones consecutivas.

    En este algoritmo podemos esperar que, después de tiempos suficientemente largos,
    las configuraciones pierdan la correlación con la condición inicial. Sin embargo,
    no podemos afirmar automáticamente que las configuraciones medidas consecutivamente
    sean independientes entre sí.
    """


    for conf in configurations:  # Comparo la configuración actual con cada configuración de referencia, de la misma forma que en el muestreo directo

        condition_hit = True  # Supongo inicialmente que la configuración actual coincide, dentro de la tolerancia, con la configuración de referencia

        for b in conf:

            condition_b = min(
                max(abs(a[0] - b[0]), abs(a[1] - b[1]))
                for a in pos
            ) < del_xy  # Reviso si existe al menos un disco simulado suficientemente cercano al disco de referencia b

            condition_hit *= condition_b  # La configuración completa cumple únicamente si todos los discos de referencia encuentran una posición suficientemente cercana

        if condition_hit:

            hits[conf] += 1  # Si se cumple la condición completa, registro una aproximación a esta configuración de referencia


    t += next_event  # Me sitúo temporalmente en el instante exacto de la siguiente colisión

    del_t = t - t_previous  # Calculo el intervalo de tiempo restante desde el último tiempo procesado hasta el instante exacto de la colisión


    for k, l in singles:

        pos[k][l] += vel[k][l] * del_t  # Muevo todos los discos hasta sus posiciones exactas en el instante de la colisión


    if min(wall_times) < min(pair_times):  # Determino si la siguiente colisión ocurre con una pared o entre dos discos

        collision_disk, direction = singles[wall_times.index(next_event)]  # Recupero qué disco colisionó con la pared y en qué dirección ocurrió la colisión

        vel[collision_disk][direction] *= -1.0  # Como las paredes están alineadas con los ejes, la colisión invierte únicamente la componente de la velocidad normal a la pared


    else:  # Colisión entre dos discos

        a, b = pairs[pair_times.index(next_event)]  # Identifico el par de discos que colisionaron

        del_x = [pos[b][0] - pos[a][0], pos[b][1] - pos[a][1]]  # Calculo el vector de posición relativa que apunta desde el disco a hacia el disco b

        abs_x = math.sqrt(del_x[0] ** 2 + del_x[1] ** 2)  # Calculo la norma del vector de posición relativa

        e_perp = [c / abs_x for c in del_x]  # Construyo el vector unitario normal a la superficie de contacto, dirigido desde a hacia b

        del_v = [vel[b][0] - vel[a][0], vel[b][1] - vel[a][1]]  # Calculo la velocidad del disco b relativa al disco a

        scal = del_v[0] * e_perp[0] + del_v[1] * e_perp[1]  # Proyecto la velocidad relativa sobre la dirección normal; como los discos se están acercando, esta cantidad es negativa


        for k in range(2):

            vel[a][k] += e_perp[k] * scal  # Actualizo la velocidad de a en la dirección normal; como scal < 0, su cambio apunta en dirección opuesta a e_perp

            vel[b][k] -= e_perp[k] * scal  # Actualizo la velocidad de b con un cambio igual y opuesto al de a, conservando el momento lineal


for conf in configurations:

    print(conf, hits[conf])

((0.3, 0.3), (0.3, 0.7), (0.7, 0.3), (0.7, 0.7)) 668
((0.2, 0.2), (0.2, 0.8), (0.75, 0.25), (0.75, 0.75)) 1655
((0.3, 0.2), (0.3, 0.8), (0.7, 0.2), (0.7, 0.7)) 1487
